# MongoDB Atlas — Create a Collection with Voyage AI Embeddings

This notebook walks through a complete end-to-end pipeline for building semantic search on MongoDB Atlas using Voyage AI embedding models:

1. **Install dependencies** — `pymongo`, `voyageai`, `python-dotenv`
2. **Load secrets** — Atlas Model API key and MongoDB URI from a `.env` file via `python-dotenv`
3. **Configure the embedding client** — Direct HTTP calls to the MongoDB Atlas Voyage AI endpoint (`https://ai.mongodb.com/v1`) using `voyage-4-large`
4. **Connect to MongoDB Atlas** — Establish a PyMongo connection and target the `voyage_demo.tech_articles` collection
5. **Prepare sample documents** — Eight data-platform / ML engineering articles across `data_engineering`, `data_governance`, and `ai_ml` categories
6. **Generate embeddings** — Batch-embed all document texts via `voyage-4-large` (1024 dimensions, `input_type="document"`)
7. **Insert documents** — Attach embeddings to each document and insert into the Atlas collection
8. **Create a Vector Search index** — Option A (Atlas UI JSON editor) or Option B (PyMongo `SearchIndexModel`) — HNSW index with cosine similarity and a `category` filter field
9. **Poll until index is ready** — Retry loop checking index status until it reaches `READY`
10. **Run vector similarity searches** — Semantic search with and without a `category` pre-filter, using `input_type="query"` embeddings
11. **Inspect the collection** — Document count, sample document preview, and embedding dimension confirmation
12. **Verify embedding normalization** — Confirm L2 norm ≈ 1.0 per vector and validate that dot product == cosine similarity for Voyage AI embeddings; includes a reference table for all three Atlas similarity metrics
13. **Delete the Vector Search index** *(optional cleanup)* — Programmatically drop the HNSW index via PyMongo with async polling
14. **Drop the collection** *(optional cleanup)* — Remove all documents and close the connection

---

> **Prerequisites**
> - MongoDB Atlas cluster (M0 free tier works)
> - **Atlas Model API key** — Atlas UI → AI Models → Create model API key *(authenticates against `https://ai.mongodb.com/v1`, not the native Voyage AI endpoint)*
> - **Atlas connection string** — Atlas UI → Connect → Drivers
> - *(Optional — Cell 8 Option B only)* Atlas Admin API public + private key pair for programmatic index creation via PyMongo


## Cell 1 — Install Dependencies

In [1]:
# !pip install pymongo voyageai python-dotenv -q

## Cell 2 — Store Secrets Safely

Create a `.env` file in the same directory as this notebook with the following content:

```
ATLAS_MODEL_API_KEY=your_atlas_model_api_key_here
MONGODB_URI=mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority
```

> ⚠️ Add `.env` to your `.gitignore` — never commit secrets.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ATLAS_MODEL_API_KEY = os.getenv("ATLAS_MODEL_API_KEY")
MONGODB_URI         = os.getenv("MONGODB_URI")

assert ATLAS_MODEL_API_KEY, "ATLAS_MODEL_API_KEY not set in .env"
assert MONGODB_URI,         "MONGODB_URI not set in .env"

print("✅ Secrets loaded successfully")

✅ Secrets loaded successfully


## Cell 3 — Configure the Voyage AI Client

The **Atlas Model API key** authenticates against `https://ai.mongodb.com/v1` — not the default Voyage AI endpoint.
We override `api_base` to point to the correct URL.

In [3]:
import requests
import time

def get_embeddings(
    texts: list[str],
    input_type: str = "document",
    max_retries: int = 5,
    base_delay: float = 2.0,
) -> list[list[float]]:
    """
    Call the MongoDB Atlas Voyage AI endpoint directly.
    Atlas Model API keys authenticate against ai.mongodb.com — not api.voyageai.com.

    Retries automatically on HTTP 429 (Too Many Requests) using exponential
    backoff.  Honours the Retry-After response header when present so we wait
    exactly as long as the server requests rather than guessing.

    Args:
        texts:       List of strings to embed (max 1 000 per request).
        input_type:  "document" when indexing; "query" at search time.
        max_retries: Maximum number of retry attempts before raising.
        base_delay:  Initial backoff delay in seconds (doubles each attempt).
    """
    for attempt in range(max_retries + 1):
        response = requests.post(
            "https://ai.mongodb.com/v1/embeddings",
            headers={
                "Authorization": f"Bearer {ATLAS_MODEL_API_KEY}",
                "Content-Type":  "application/json"
            },
            json={
                "model"      : "voyage-4-large",
                "input"      : texts,
                "input_type" : input_type
            }
        )

        if response.status_code == 429:
            if attempt == max_retries:
                response.raise_for_status()          # exhausted — bubble up

            # Respect Retry-After header if the server sends one; otherwise back off
            retry_after = response.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else base_delay * (2 ** attempt)
            print(f"   ⏳ Rate limited (429). Waiting {wait:.1f}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait)
            continue                                 # retry

        response.raise_for_status()                  # raise on any other 4xx/5xx
        data = response.json()
        # Sort by index to preserve order, then extract vectors
        return [item["embedding"] for item in sorted(data["data"], key=lambda x: x["index"])]

# Quick sanity check
test_emb = get_embeddings(["hello from MongoDB Atlas"])
print(f"✅ Atlas endpoint working. Embedding dims: {len(test_emb[0])}")


✅ Atlas endpoint working. Embedding dims: 1024


## Cell 4 — Connect to MongoDB Atlas

In [4]:
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

# Confirm connection
client.admin.command("ping")
print("✅ Connected to MongoDB Atlas")

# --------------------------------------------------
# Configure your target database and collection here
# --------------------------------------------------
DB_NAME         = "voyage_demo"
COLLECTION_NAME = "tech_articles"
EMBEDDING_FIELD = "embedding"         # field that will hold the vector
EMBEDDING_DIMS  = 1024                # voyage-4-large default output dims

db         = client[DB_NAME]
collection = db[COLLECTION_NAME]

print(f"📦 Target: {DB_NAME}.{COLLECTION_NAME}")

✅ Connected to MongoDB Atlas
📦 Target: voyage_demo.tech_articles


## Cell 5 — Sample Dataset

A small corpus of data platform / ML engineering blurbs — realistic enough to test semantic search.

In [5]:
sample_docs = [
    {
        "title": "Delta Lake ACID Transactions",
        "category": "data_engineering",
        "text": "Delta Lake brings ACID transaction support to Apache Spark and big data workloads. It enables reliable data lakes with features like schema enforcement, time travel, and upsert operations via MERGE INTO."
    },
    {
        "title": "Databricks Unity Catalog",
        "category": "data_governance",
        "text": "Unity Catalog is Databricks' unified governance layer for data and AI assets. It provides centralized access control, lineage tracking, and auditing across all workspaces in an account."
    },
    {
        "title": "Vector Search and RAG Pipelines",
        "category": "ai_ml",
        "text": "Retrieval-Augmented Generation (RAG) uses vector similarity search to fetch semantically relevant documents at query time. Embedding models convert text into dense vectors, and approximate nearest neighbour search retrieves the closest matches."
    },
    {
        "title": "MongoDB Atlas Vector Search",
        "category": "ai_ml",
        "text": "MongoDB Atlas Vector Search enables storing and querying high-dimensional vector embeddings alongside your operational data. It supports approximate nearest neighbour (ANN) search using the HNSW algorithm."
    },
    {
        "title": "Lakehouse Architecture Overview",
        "category": "data_engineering",
        "text": "A Lakehouse combines the scalability and cost efficiency of a data lake with the data management and ACID guarantees of a data warehouse. Formats like Delta Lake and Apache Iceberg underpin modern Lakehouse designs."
    },
    {
        "title": "Snowflake Cortex Fine-Tuning",
        "category": "ai_ml",
        "text": "Snowflake Cortex Fine-tuning allows fine-tuning LLMs on your own data directly within Snowflake using SQL. It is serverless, requires no GPU management, and supports models like Mistral and Llama."
    },
    {
        "title": "Medallion Architecture",
        "category": "data_engineering",
        "text": "The Medallion Architecture organises data into Bronze (raw ingestion), Silver (cleansed and validated), and Gold (aggregated and business-ready) layers. It is a best practice pattern for Delta Lake and Databricks Lakehouses."
    },
    {
        "title": "Voyage AI Embedding Models",
        "category": "ai_ml",
        "text": "Voyage AI, now part of MongoDB, provides state-of-the-art embedding and reranking models. The Voyage 4 series includes voyage-4-large, voyage-4, voyage-4-lite, and voyage-4-nano, all sharing the same embedding space for flexible retrieval."
    }
]

print(f"📄 {len(sample_docs)} sample documents ready")

📄 8 sample documents ready


## Cell 6 — Generate Embeddings

We embed all document texts in a single batch call (Voyage supports up to 1,000 texts per request).

In [6]:
texts = [doc["text"] for doc in sample_docs]

embeddings = get_embeddings(texts, input_type="document")

# Attach embeddings back to documents
for doc, emb in zip(sample_docs, embeddings):
    doc[EMBEDDING_FIELD] = emb

print(f"✅ Generated {len(embeddings)} embeddings")
print(f"   Model     : voyage-4-large")
print(f"   Dimensions: {len(embeddings[0])}")

✅ Generated 8 embeddings
   Model     : voyage-4-large
   Dimensions: 1024


## Cell 7 — Insert Documents into Atlas Collection

In [7]:
# Drop and recreate for a clean demo run — remove this in production!
collection.drop()
print(f"🗑️  Dropped existing '{COLLECTION_NAME}' collection (fresh start)")

insert_result = collection.insert_many(sample_docs)

print(f"✅ Inserted {len(insert_result.inserted_ids)} documents into '{DB_NAME}.{COLLECTION_NAME}'")
print(f"   First _id: {insert_result.inserted_ids[0]}")

🗑️  Dropped existing 'tech_articles' collection (fresh start)
✅ Inserted 8 documents into 'voyage_demo.tech_articles'
   First _id: 69e47761a9ce4561cef0dea2


## Cell 8 — Create a Vector Search Index

This uses the **Atlas Search Management API** to create an HNSW vector index programmatically.

> ⚠️ **Note:** You need an **Atlas API Public Key + Private Key** (different from the Model API key) to call the Atlas Admin API. 
> Alternatively, create the index from the Atlas UI: **Browse Collections → Search Indexes → Create Search Index → JSON Editor**.

### Option A — Atlas UI (Recommended for beginners)

Paste this JSON when creating a Vector Search index in the Atlas UI:

```json
{
  "fields": [
    {
      "type": "vector",
      "path": "embedding",
      "numDimensions": 1024,
      "similarity": "cosine"
    },
    {
      "type": "filter",
      "path": "category"
    }
  ]
}
```

### Option B — Programmatic (PyMongo 4.7+ with Atlas cluster)

In [8]:
import time
from pymongo.operations import SearchIndexModel

INDEX_NAME = "vector_index"

# Define the vector search index
index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type"          : "vector",
                "path"          : EMBEDDING_FIELD,
                "numDimensions" : EMBEDDING_DIMS,
                "similarity"    : "cosine"      # cosine works well for normalised Voyage embeddings
            },
            {
                "type": "filter",
                "path": "category"              # enables pre-filter by category during search
            }
        ]
    },
    name = INDEX_NAME,
    type = "vectorSearch"
)

# Create the index (non-blocking — Atlas builds it asynchronously)
try:
    collection.create_search_index(model=index_model)
    print(f"✅ Vector Search index '{INDEX_NAME}' creation requested")
    print("   ⏳ Atlas is building the index — this takes ~30–60 seconds on a free tier cluster")
    print("   Run the next cell after the index shows as 'Active' in the Atlas UI")
except Exception as e:
    print(f"⚠️  Index creation error (may already exist): {e}")

✅ Vector Search index 'vector_index' creation requested
   ⏳ Atlas is building the index — this takes ~30–60 seconds on a free tier cluster
   Run the next cell after the index shows as 'Active' in the Atlas UI


## Cell 9 — Poll Until Index is Ready

In [9]:
import time

print("⏳ Waiting for index to become READY...")

for attempt in range(20):  # poll up to ~100 seconds
    indexes = list(collection.list_search_indexes(INDEX_NAME))
    if indexes and indexes[0].get("status") == "READY":
        print(f"✅ Index '{INDEX_NAME}' is READY after ~{attempt * 5}s")
        break
    status = indexes[0].get("status", "PENDING") if indexes else "PENDING"
    print(f"   [{attempt+1}/20] Status: {status} — retrying in 5s...")
    time.sleep(5)
else:
    print("⚠️  Index not ready yet. Wait a bit longer and re-run the search cell.")

⏳ Waiting for index to become READY...
   [1/20] Status: PENDING — retrying in 5s...
   [2/20] Status: PENDING — retrying in 5s...
   [3/20] Status: PENDING — retrying in 5s...
   [4/20] Status: PENDING — retrying in 5s...
   [5/20] Status: PENDING — retrying in 5s...
   [6/20] Status: PENDING — retrying in 5s...
✅ Index 'vector_index' is READY after ~30s


## Cell 10 — Run a Vector Similarity Search

In [10]:
def vector_search(query_text: str, top_k: int = 3, filter_category: str = None):
    """Embed a query and retrieve the top_k most semantically similar documents."""

    # Embed the query — use input_type='query' for search queries
    query_embedding = get_embeddings([query_text], input_type="query")[0]

    # Build the $vectorSearch stage
    vector_search_stage = {
        "$vectorSearch": {
            "index"        : INDEX_NAME,
            "path"         : EMBEDDING_FIELD,
            "queryVector"  : query_embedding,
            "numCandidates": top_k * 10,
            "limit"        : top_k
        }
    }

    # Optional pre-filter on category
    if filter_category:
        vector_search_stage["$vectorSearch"]["filter"] = {"category": {"$eq": filter_category}}

    pipeline = [
        vector_search_stage,
        {
            "$project": {
                "_id"     : 0,
                "title"   : 1,
                "category": 1,
                "text"    : 1,
                "score"   : {"$meta": "vectorSearchScore"}
            }
        }
    ]

    return list(collection.aggregate(pipeline))


# ── Test Query 1: General semantic search ──────────────────────────────────
print("=" * 60)
print("Query: 'How does data governance work in modern data platforms?'")
print("=" * 60)
results = vector_search("How does data governance work in modern data platforms?", top_k=3)
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']} ({r['category']})")
    print(f"          {r['text'][:100]}...")

# Brief pause between queries — avoids back-to-back 429s on the Atlas Model API
time.sleep(1)

# ── Test Query 2: Category-filtered search ─────────────────────────────────
print("=" * 60)
print("Query: 'vector embeddings for similarity search' | Filter: ai_ml")
print("=" * 60)
results = vector_search("vector embeddings for similarity search", top_k=3, filter_category="ai_ml")
for r in results:
    print(f"  [{r['score']:.4f}] {r['title']} ({r['category']})")
    print(f"          {r['text'][:100]}...")


Query: 'How does data governance work in modern data platforms?'
  [0.7180] Databricks Unity Catalog (data_governance)
          Unity Catalog is Databricks' unified governance layer for data and AI assets. It provides centralize...
  [0.6802] Medallion Architecture (data_engineering)
          The Medallion Architecture organises data into Bronze (raw ingestion), Silver (cleansed and validate...
  [0.6574] Delta Lake ACID Transactions (data_engineering)
          Delta Lake brings ACID transaction support to Apache Spark and big data workloads. It enables reliab...
Query: 'vector embeddings for similarity search' | Filter: ai_ml
   ⏳ Rate limited (429). Waiting 2.0s before retry 1/5...
   ⏳ Rate limited (429). Waiting 4.0s before retry 2/5...
   ⏳ Rate limited (429). Waiting 8.0s before retry 3/5...
  [0.7683] Voyage AI Embedding Models (ai_ml)
          Voyage AI, now part of MongoDB, provides state-of-the-art embedding and reranking models. The Voyage...
  [0.7593] Vector Search and

## Cell 11 — Inspect the Collection

In [11]:
import json

# Count docs
print(f"Total documents : {collection.count_documents({})}")

# Preview one document (without the full embedding vector)
sample = collection.find_one({}, {"embedding": 0})
print("\nSample document (embedding hidden):")
print(json.dumps({k: v for k, v in sample.items() if k != "_id"}, indent=2))

# Show embedding dimension of first doc
doc_with_emb = collection.find_one({}, {"embedding": 1})
print(f"\nEmbedding dimensions: {len(doc_with_emb['embedding'])}")

Total documents : 8

Sample document (embedding hidden):
{
  "title": "Delta Lake ACID Transactions",
  "category": "data_engineering",
  "text": "Delta Lake brings ACID transaction support to Apache Spark and big data workloads. It enables reliable data lakes with features like schema enforcement, time travel, and upsert operations via MERGE INTO."
}

Embedding dimensions: 1024


## Cell 12 — Similarity Metric Reference

MongoDB Atlas Vector Search supports three similarity metrics — set at **index creation time** and cannot be changed without dropping and recreating the index.

| Metric | Value | Best For |
|---|---|---|
| Cosine | `"cosine"` | General text/semantic search — measures angle between vectors, ignores magnitude |
| Dot Product | `"dotProduct"` | Unit-normalized embeddings (e.g. Voyage AI, OpenAI) — fastest, identical ranking to cosine on normalized vectors |
| Euclidean | `"euclidean"` | Spatial or numerical data — measures absolute distance in vector space |

> **Recommendation for Voyage AI:** Use `"dotProduct"` — Voyage embeddings are L2-normalized, so dot product and cosine produce identical rankings with lower compute cost.

In [12]:
import numpy as np

print("=" * 60)
print("VERIFICATION: Are stored embeddings normalized?")
print("=" * 60)

# Fetch a few documents with their embeddings
docs = list(collection.find({}, {"title": 1, "embedding": 1}).limit(4))

for doc in docs:
    emb   = np.array(doc["embedding"])
    norm  = np.linalg.norm(emb)           # L2 norm — should be ~1.0 if normalized
    mean  = np.mean(emb)
    stdev = np.std(emb)

    status = "✅ Normalized" if abs(norm - 1.0) < 1e-3 else "⚠️  NOT normalized"

    print(f"\n  {status} | {doc['title']}")
    print(f"    L2 Norm  : {norm:.6f}   ← should be ~1.0 for normalized vectors")
    print(f"    Mean     : {mean:.6f}")
    print(f"    Std Dev  : {stdev:.6f}")
    print(f"    Dims     : {len(emb)}")

# Cross-check: manual cosine similarity vs dot product
# For normalized vectors these should be identical
print("\n" + "=" * 60)
print("CROSS-CHECK: cosine similarity == dot product for normalized vectors?")
print("=" * 60)

emb_a = np.array(docs[0]["embedding"])
emb_b = np.array(docs[1]["embedding"])

dot_product    = np.dot(emb_a, emb_b)
cosine_sim     = np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))
are_equivalent = abs(dot_product - cosine_sim) < 1e-5

print(f"  Doc A : {docs[0]['title']}")
print(f"  Doc B : {docs[1]['title']}")
print(f"  Dot Product      : {dot_product:.6f}")
print(f"  Cosine Similarity: {cosine_sim:.6f}")
print(f"  Are equivalent?  : {'✅ Yes' if are_equivalent else '⚠️  No — vectors may not be normalized'}")

VERIFICATION: Are stored embeddings normalized?

  ✅ Normalized | Delta Lake ACID Transactions
    L2 Norm  : 1.000000   ← should be ~1.0 for normalized vectors
    Mean     : -0.001363
    Std Dev  : 0.031220
    Dims     : 1024

  ✅ Normalized | Databricks Unity Catalog
    L2 Norm  : 1.000000   ← should be ~1.0 for normalized vectors
    Mean     : -0.001415
    Std Dev  : 0.031218
    Dims     : 1024

  ✅ Normalized | Vector Search and RAG Pipelines
    L2 Norm  : 1.000000   ← should be ~1.0 for normalized vectors
    Mean     : -0.001797
    Std Dev  : 0.031198
    Dims     : 1024

  ✅ Normalized | MongoDB Atlas Vector Search
    L2 Norm  : 1.000000   ← should be ~1.0 for normalized vectors
    Mean     : -0.001575
    Std Dev  : 0.031210
    Dims     : 1024

CROSS-CHECK: cosine similarity == dot product for normalized vectors?
  Doc A : Delta Lake ACID Transactions
  Doc B : Databricks Unity Catalog
  Dot Product      : 0.664012
  Cosine Similarity: 0.664012
  Are equivalent?  : 

## Cell 13 — Delete the Vector Search Index (Cleanup Optional)

Programmatically drop the Vector Search index created in Cell 8 using PyMongo.
Useful for resetting the index definition (e.g. changing similarity metric) or full cleanup.

> ⚠️ Dropping the index does **not** delete the documents in the collection — only the search index is removed.
> To also drop the collection, uncomment the corresponding line in Cell 12.

In [13]:
# import time

# print(f"🗑️  Dropping Vector Search index '{INDEX_NAME}'...")

# try:
#     collection.drop_search_index(INDEX_NAME)

#     # Poll until the index is fully deleted
#     # Atlas deletes indexes asynchronously — the index may briefly still appear
#     for attempt in range(20):
#         existing = [
#             idx for idx in collection.list_search_indexes()
#             if idx.get("name") == INDEX_NAME
#         ]
#         if not existing:
#             print(f"✅ Index '{INDEX_NAME}' successfully deleted after ~{attempt * 3}s")
#             break
#         status = existing[0].get("status", "DELETING")
#         print(f"   [{attempt+1}/20] Status: {status} — waiting 3s...")
#         time.sleep(3)
#     else:
#         print("⚠️  Index deletion is taking longer than expected. Check Atlas UI to confirm.")

# except Exception as e:
#     print(f"❌ Error deleting index: {e}")

## Cell 14 — Drop the collection (Cleanup Optional)

In [14]:
# Uncomment to drop the collection and close the connection
# collection.drop()
# client.close()
# print("🧹 Collection dropped and connection closed")